# HIGHFLY · PartCrafter gratuito · Prueba de armadura 3D (R0.4)

**Objetivo:** recibir una imagen de referencia de una armadura y generar piezas 3D reales usando PartCrafter. El resultado será un ZIP con `object.glb`, `part_XX.glb`, manifiesto y procedencia. **Todavía NO es una armadura equipable en ClaudeCraft**: debe pasar por la adaptación Blender y revisión visual del laboratorio.

**Costo:** este cuaderno no contrata GPU, API ni suscripciones. Requiere que Google Colab asigne **una GPU gratuita CUDA con al menos 8 GB de VRAM**, algo que Colab no garantiza. No ejecutar ni elegir opciones de pago. No ejecutar de manera automatizada/desatendida.

**Seguridad/licencias:** código PartCrafter y pesos principales MIT; no se usan API de pago ni generación VLM. Para evitar el modelo RMBG-1.4 (uso no comercial), el cuaderno **omite su descarga y uso**; prepararemos la imagen sobre fondo blanco mediante Pillow. No uses este prototipo como garantía de un producto comercial listo.

**Cómo usar:** 1) seleccionar GPU gratuita en Entorno de ejecución > Cambiar tipo de entorno de ejecución; 2) ejecutar las celdas en orden; 3) cargar imagen; 4) obtener ZIP; 5) enviar el ZIP a Guine para integrar con Blender/GitHub. Solo subir imágenes que puedas usar.


In [ ]:
import os, sys, subprocess, platform
import torch
print("Python",sys.version.split()[0],"Torch",torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("Colab NO asignó CUDA gratis. Revisá Entorno de ejecución > GPU. No contrates nada.")
props=torch.cuda.get_device_properties(0)
gb=props.total_memory / (1024**3)
print("GPU:",props.name, "| VRAM:",round(gb,2),"GiB")
if gb<7.5:
    raise RuntimeError("PartCrafter requiere al menos 8GB VRAM. No se ejecutará una inferencia imposible.")
assert os.environ.get("GEMINI_API_KEY") is None, "Para esta prueba no hacen falta claves de IA de pago"
print("GPU GRATIS compatible en principio; Colab puede cortar la sesión o agotar recursos.")


## 1. Instalar PartCrafter oficial

Este paso descarga dependencias, pesos del modelo abierto y puede demorar varios minutos. Usamos un commit fijo de GitHub para no cambiar la implementación de un día al otro. **Si la instalación falla, detenete**; no hay pagos ni reintentos automáticos.


In [ ]:
import os, sys, subprocess
from pathlib import Path
def run(cmd,cwd=None,timeout=900):
    print(">", " ".join(map(str,cmd)))
    subprocess.run(list(map(str,cmd)),cwd=cwd,check=True,timeout=timeout)
root=Path("/content/PartCrafter")
if not root.exists():
    run(["git","clone","--no-checkout","https://github.com/wgsxm/PartCrafter.git",root])
run(["git","checkout","3d773bf02fad51c7ab31a5615573fec93b287b30"],cwd=root)
# Dependencias CUDA de la receta oficial, sin cuentas/API de pago.
run([sys.executable,"-m","pip","install","-q","torch==2.5.1","torchvision==0.20.1","torchaudio==2.5.1",
     "--index-url","https://download.pytorch.org/whl/cu124"],timeout=1200)
run(["bash","settings/setup.sh"],cwd=root,timeout=1500)
print("PartCrafter instalado. Si PyTorch pidió reiniciar sesión, reiniciá y volvé a ejecutar desde el inicio.")


## 2. Evitar dependencia RMBG no comercial y preparar imagen

PartCrafter carga RMBG aun cuando no se pide remover fondo. Esta celda hace un cambio acotado y comprobado para omitir esa dependencia. La imagen se centra sobre **fondo blanco**, sin IA adicional. Es una prueba: ilustraciones con mucho fondo o una figura entera pueden producir piezas que no sean vestimentas.


In [ ]:
from pathlib import Path
from PIL import Image,ImageOps
from google.colab import files
src=(root/"scripts/inference_partcrafter.py")
txt=src.read_text(encoding="utf-8")
old_a='    snapshot_download(repo_id="briaai/RMBG-1.4", local_dir=rmbg_weights_dir)'
new_a='    if args.rmbg:\n        snapshot_download(repo_id="briaai/RMBG-1.4", local_dir=rmbg_weights_dir)'
old_b='    rmbg_net = BriaRMBG.from_pretrained(rmbg_weights_dir).to(device)\n    rmbg_net.eval()'
new_b='    rmbg_net = None\n    if args.rmbg:\n        rmbg_net = BriaRMBG.from_pretrained(rmbg_weights_dir).to(device)\n        rmbg_net.eval()'
if old_a in txt and old_b in txt:
    txt=txt.replace(old_a,new_a).replace(old_b,new_b)
    src.write_text(txt,encoding="utf-8")
elif new_a not in txt or new_b not in txt:
    raise RuntimeError("La fuente PartCrafter cambió. Se detiene antes de usar RMBG; revisar implementación.")
uploaded=files.upload()
if len(uploaded)!=1:raise ValueError("Subí UNA imagen PNG/JPG de armadura, no archivos privados ni múltiples imágenes.")
name,next_data=next(iter(uploaded.items()))
if not name.lower().endswith((".png",".jpg",".jpeg",".webp")):
    raise ValueError("Solo imagen PNG/JPEG/WEBP")
from io import BytesIO
image=Image.open(BytesIO(next_data)).convert("RGBA")
# Mantener silueta y evitar IA RMBG: transparencia verdadera si estaba en PNG.
background=Image.new("RGBA",image.size,(255,255,255,255))
background.alpha_composite(image)
image=background.convert("RGB")
image.thumbnail((768,768))
canvas=Image.new("RGB",(768,768),"white")
canvas.paste(image,((768-image.width)//2,(768-image.height)//2))
prepared=Path("/content/highfly_obsidian_reference.png")
canvas.save(prepared)
display(canvas.resize((384,384)))
print("Imagen lista (no hay bodypaint ni recolor):",prepared)


## 3. Generar geometrías independientes (si Colab dispone de GPU suficiente)

Se generan 4 partes para comenzar y ahorrar VRAM. Podés probar 6 después, pero no hay promesa de fidelidad artística, posiciones correctas ni identificación automática de casco/torso/brazos. **Nunca llamamos Gemini, Meshy u otro servicio pago.**


In [ ]:
from pathlib import Path
import subprocess,sys,os
tag="highfly_obsidian_pilot"
outdir=Path("/content/highfly_partcrafter_output")
cmd=[sys.executable,"scripts/inference_partcrafter.py",
 "--image_path",str(prepared),"--num_parts","4",
 "--num_tokens","512","--num_inference_steps","25",
 "--output_dir",str(outdir),"--tag",tag,"--seed","17"]
assert "--rmbg" not in cmd and "--part_suggest" not in cmd and "--style_transfer" not in cmd
subprocess.run(cmd,cwd=root,check=True,timeout=2400,env=dict(os.environ,GEMINI_API_KEY=""))
target=outdir/tag
assert (target/"object.glb").is_file(), "PartCrafter no produjo GLB, nada para importar."
pieces=sorted(target.glob("part_*.glb"))
assert len(pieces)==4 and all(x.stat().st_size>1000 for x in pieces)
print("GLB PARTES:",[(p.name,p.stat().st_size) for p in pieces])


## 4. Comprobar archivos y descargar ZIP para HIGHFLY

El ZIP contiene geometrías generadas y procedencia, sin credenciales. El siguiente paso es **auditar calidad, asignar piezas a slots y pesar en Blender**. No lo vamos a publicar como armadura lista.


In [ ]:
import json, zipfile,hashlib,datetime
from pathlib import Path
from google.colab import files
source_files=[target/"object.glb",target/"manifest.json",*sorted(target.glob("part_*.glb"))]
for p in source_files:
    assert p.is_file() and p.stat().st_size>30, f"Falta {p.name}"
meta={
 "schema":"highfly.partcrafter.raw-3d/v1",
 "provider":"PartCrafter MIT - image to multi-part 3D",
 "repository":"wgsxm/PartCrafter",
 "commit":"3d773bf02fad51c7ab31a5615573fec93b287b30",
 "gpu":torch.cuda.get_device_name(0),
 "input":"user-supplied image, white background; NO RMBG, NO paid VLM",
 "generatedParts":4,"riggedToClaudeCraft":False,"artApproved":False,
 "fileSha256":{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in source_files}}
meta_file=target/"highfly-provenance.json"
meta_file.write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding="utf-8")
zip_path=Path("/content/HIGHFLY_OBSIDIAN_PARTCRAFTER_RAW3D.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in [*source_files,meta_file]:z.write(p,arcname=p.name)
print("Archivo real:",zip_path, "(",zip_path.stat().st_size,"bytes )")
files.download(str(zip_path))
